# 02 - RAG Pipeline

This notebook builds the Retrieval-Augmented Generation (RAG) pipeline
for the CarCare AI automotive maintenance assistant.

The pipeline will:

1. Load the cleaned automotive maintenance documents.
2. Split the documents into searchable chunks.
3. Generate semantic embeddings for each chunk.
4. Store the embeddings in ChromaDB.
5. Retrieve relevant maintenance information for user queries.
6. Generate grounded answers using a local Ollama LLM.
7. Evaluate retrieval quality and identify failure cases.

Each chunk preserves its original PDF page number and source filename
to support transparent source citations.

The RAG pipeline uses the following architecture:

User Query
→ Embedding Model
→ ChromaDB Retrieval
→ Relevant Maintenance Context
→ Ollama LLM
→ Answer + Sources


In [20]:
from pathlib import Path
import json

# Project paths
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# Cleaned PDF data from Notebook 01
PDF_JSON_PATH = PROCESSED_DIR / "pdf_pages_cleaned.json"

if not PDF_JSON_PATH.exists():
    raise FileNotFoundError(
        f"Cleaned PDF file not found:\n{PDF_JSON_PATH}"
    )

# Load cleaned documents
with open(PDF_JSON_PATH, "r", encoding="utf-8") as f:
    cleaned_documents = json.load(f)

print(f"Loaded documents: {len(cleaned_documents)}")
print(f"PDF JSON: {PDF_JSON_PATH}")
print(f"ChromaDB directory: {CHROMA_DIR}")
print(f"PDF JSON exists: {PDF_JSON_PATH.exists()}")

Loaded documents: 145
PDF JSON: c:\Users\EGY SKY\OneDrive\Desktop\CarCare_AI\data\processed\pdf_pages_cleaned.json
ChromaDB directory: c:\Users\EGY SKY\OneDrive\Desktop\CarCare_AI\data\chroma_db
PDF JSON exists: True


## 1. Document Chunking

The cleaned PDF is organized page by page.

For semantic retrieval, each page is split into smaller overlapping chunks.

Each chunk preserves:

* A unique chunk ID.
* The original PDF page number.
* The source PDF filename.
* The chunk text.

Overlapping chunks help preserve context when important information
spans across chunk boundaries.


In [21]:
# Chunk configuration
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

chunks = []

for document in cleaned_documents:
    text = document["text"].strip()

    # Skip empty pages
    if not text:
        continue

    page = document["page"]
    source = document["source"]

    start = 0
    chunk_number = 0

    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "chunk_id": f"page_{page}_chunk_{chunk_number}",
                "text": chunk_text,
                "page": page,
                "source": source
            })

        if end >= len(text):
            break

        start = end - CHUNK_OVERLAP
        chunk_number += 1

print(f"Total chunks: {len(chunks)}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Chunk overlap: {CHUNK_OVERLAP}")

print("\nSample chunk:")
print(chunks[0]["text"][:700])
print(f"\nPage: {chunks[0]['page']}")
print(f"Source: {chunks[0]['source']}")

Total chunks: 302
Chunk size: 1000
Chunk overlap: 150

Sample chunk:
Vehicle Maintenance and Repair Series
Vehicle
Maintenance
Vehicle Fitting Units Level 1 and 2
Roy Brooks, Jack Hirst, John Whipp
Australia • Canada • Mexico • Singapore • Spain • United Kingdom • United States
www.TechnicalBooksPDF.com

Page: 2
Source: Vehicle Maintenance Vehicle Fitting Units Level 1 and 2 by Roy Brooks, Jack Hirst and John Whipp.pdf


In [22]:
# Remove unwanted website references from the chunks

def clean_chunk_text(text):
    # Remove the extracted TechnicalBooksPDF URL
    text = re.sub(r"\[www\.TechnicalBooksPDF\.com\]\([^)]*\)", "", text)
    text = re.sub(r"https?://\S+", "", text)

    # Normalize extra spaces
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()


for chunk in chunks:
    chunk["text"] = clean_chunk_text(chunk["text"])

print(f"Total chunks after cleaning: {len(chunks)}")

print("\nCleaned sample chunk:")
print(chunks[0]["text"][:700])
print(f"\nPage: {chunks[0]['page']}")

Total chunks after cleaning: 302

Cleaned sample chunk:
Vehicle Maintenance and Repair Series
Vehicle
Maintenance
Vehicle Fitting Units Level 1 and 2
Roy Brooks, Jack Hirst, John Whipp
Australia • Canada • Mexico • Singapore • Spain • United Kingdom • United States
www.TechnicalBooksPDF.com

Page: 2


In [23]:
# Remove unwanted website references from the chunks

def clean_chunk_text(text):
    # Remove Markdown-style links
    text = re.sub(r"\[([^\]]*TechnicalBooksPDF[^\]]*)\]\([^)]*\)", "", text)

    # Remove plain website references
    text = re.sub(r"www\.TechnicalBooksPDF\.com", "", text, flags=re.IGNORECASE)

    # Remove URLs
    text = re.sub(r"https?://\S+", "", text)

    # Normalize extra spaces
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()


for chunk in chunks:
    chunk["text"] = clean_chunk_text(chunk["text"])

print(f"Total chunks: {len(chunks)}")

print("\nCleaned sample chunk:")
print(chunks[0]["text"][:700])
print(f"\nPage: {chunks[0]['page']}")

Total chunks: 302

Cleaned sample chunk:
Vehicle Maintenance and Repair Series
Vehicle
Maintenance
Vehicle Fitting Units Level 1 and 2
Roy Brooks, Jack Hirst, John Whipp
Australia • Canada • Mexico • Singapore • Spain • United Kingdom • United States

Page: 2


In [24]:
# Save processed PDF chunks

CHUNKS_JSON_PATH = PROCESSED_DIR / "pdf_chunks.json"

with open(CHUNKS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Saved: {CHUNKS_JSON_PATH}")
print(f"File exists: {CHUNKS_JSON_PATH.exists()}")
print(f"Chunks saved: {len(chunks)}")

Saved: c:\Users\EGY SKY\OneDrive\Desktop\CarCare_AI\data\processed\pdf_chunks.json
File exists: True
Chunks saved: 302


## 2. Semantic Embeddings

Each document chunk is converted into a numerical vector using a
Sentence-Transformers embedding model.

The embedding vectors allow the system to compare the semantic meaning
of a user's question with the maintenance knowledge stored in the PDF.

Model:

* `all-MiniLM-L6-v2`
* Embedding dimension: 384
* Normalized embeddings for cosine-similarity retrieval


In [25]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3787.80it/s]


Embedding model: all-MiniLM-L6-v2
Embedding dimension: 384


C:\Users\EGY SKY\AppData\Local\Temp\ipykernel_25796\1108893601.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


In [26]:
# Generate embeddings for all document chunks

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding shape: {embeddings.shape}")

Batches: 100%|██████████| 10/10 [00:03<00:00,  2.73it/s]

Number of embeddings: 302
Embedding shape: (302, 384)


## 3. ChromaDB Vector Store

ChromaDB is used as the persistent vector database for the CarCare AI
knowledge base.

Each stored chunk includes:

* The chunk text.
* Its 384-dimensional embedding.
* The original PDF page number.
* The source filename.
* A unique chunk ID.

The collection is recreated from the current automotive maintenance PDF
to ensure that no data from the previous document remains.



In [27]:
import chromadb

print(f"ChromaDB version: {chromadb.__version__}")

# Create persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

COLLECTION_NAME = "carcare_maintenance"

# Delete the old collection if it exists
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")
except Exception:
    print(f"No existing collection found: {COLLECTION_NAME}")

# Create a fresh collection
collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"description": "CarCare AI automotive maintenance knowledge base"}
)

print(f"Collection created: {collection.name}")
print(f"Initial document count: {collection.count()}")

ChromaDB version: 1.5.9
Deleted existing collection: carcare_maintenance
Collection created: carcare_maintenance
Initial document count: 0


In [28]:
# Prepare data for ChromaDB

ids = [chunk["chunk_id"] for chunk in chunks]
documents = [chunk["text"] for chunk in chunks]

metadatas = [
    {
        "page": chunk["page"],
        "source": chunk["source"]
    }
    for chunk in chunks
]

# Add chunks and embeddings to ChromaDB
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print(f"Documents stored in ChromaDB: {collection.count()}")

Documents stored in ChromaDB: 302


## 4. Retrieval Testing

Before connecting the RAG pipeline to the language model, we test whether
ChromaDB can retrieve relevant maintenance information.

The test queries cover different topics from the automotive maintenance
knowledge base, including:

* Workshop safety.
* Accidents and hazards.
* Tyres.
* Electrical systems.
* Brakes.
* Suspension and steering.

The retrieved chunks are inspected together with their original PDF page
numbers.


In [29]:
# Retrieval test function

def retrieve_documents(query, top_k=5):
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    
    retrieved = []
    
    for i in range(len(results["documents"][0])):
        retrieved.append({
            "rank": i + 1,
            "text": results["documents"][0][i],
            "page": results["metadatas"][0][i]["page"],
            "source": results["metadatas"][0][i]["source"],
            "distance": results["distances"][0][i]
        })
    
    return retrieved


test_query = "What safety precautions should be followed in a vehicle workshop?"

results = retrieve_documents(test_query, top_k=5)

print(f"Query: {test_query}\n")

for result in results:
    print("=" * 80)
    print(f"Rank: {result['rank']}")
    print(f"Page: {result['page']}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Text: {result['text'][:500]}")
    print()

C:\Users\EGY SKY\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:33<00:00, 2.49MiB/s]


Query: What safety precautions should be followed in a vehicle workshop?

Rank: 1
Page: 26
Distance: 0.5856
Text: Security
Make sure that parts are kept as safe as possible. If visitors wander
around the workshop and are not observed, they could steal things.
Theft by staff may also occur.
Taking goods, equipment or money without permission is always
theft
. Theft by staff is 
gross misconduct
, and can lead to dismissal.
Do not leave keys in unattended cars. It is not uncommon to have
cars driven away from the forecourt.
Using resources economically
Avoid wasting power
• Turn off lights when they are not n

Rank: 2
Page: 11
Distance: 0.6156
Text: Hazards
A
hazard
is anything that might cause an accident or injury.
Look around your workshop. You will probably see at least one
hazard, perhaps more. Some can quickly be removed. Examples are:
• wheels left lying on the ﬂoor, after being taken off a vehicle
• oil spilt on the ﬂoor
• a trolley jack handle left lying where someone could trip

## 5. Hybrid Retrieval

Semantic retrieval is useful for understanding the meaning of a query, but
some automotive maintenance questions contain important technical keywords.

To improve retrieval reliability, the CarCare AI pipeline will combine:

* Semantic similarity from ChromaDB.
* Keyword matching for important maintenance terms.
* Metadata such as the original PDF page number.

This hybrid approach helps retrieve technically relevant maintenance
information when semantic similarity alone is not sufficiently precise.


In [30]:
# Inspect all retrieved results more compactly

for result in results:
    preview = result["text"].replace("\n", " ")
    
    print(
        f"Rank {result['rank']} | "
        f"Page {result['page']} | "
        f"Distance {result['distance']:.4f}"
    )
    print(preview[:300])
    print("-" * 80)

Rank 1 | Page 26 | Distance 0.5856
Security Make sure that parts are kept as safe as possible. If visitors wander around the workshop and are not observed, they could steal things. Theft by staff may also occur. Taking goods, equipment or money without permission is always theft . Theft by staff is  gross misconduct , and can lead to
--------------------------------------------------------------------------------
Rank 2 | Page 11 | Distance 0.6156
Hazards A hazard is anything that might cause an accident or injury. Look around your workshop. You will probably see at least one hazard, perhaps more. Some can quickly be removed. Examples are: • wheels left lying on the ﬂoor, after being taken off a vehicle • oil spilt on the ﬂoor • a trolley jac
--------------------------------------------------------------------------------
Rank 3 | Page 17 | Distance 0.6218
sockets, screwdrivers, pliers, hammers, chisels and ﬁles. To work safely with them use your common sense, know which tool to use, a

In [31]:
# Hybrid retrieval: semantic similarity + keyword matching

def hybrid_retrieve(query, top_k=5, semantic_k=10):
    # Get more semantic candidates first
    results = collection.query(
        query_texts=[query],
        n_results=semantic_k
    )

    query_words = set(
        re.findall(r"\b[a-zA-Z]{3,}\b", query.lower())
    )

    scored_results = []

    for i in range(len(results["documents"][0])):
        text = results["documents"][0][i]
        page = results["metadatas"][0][i]["page"]
        source = results["metadatas"][0][i]["source"]
        distance = results["distances"][0][i]

        text_words = set(
            re.findall(r"\b[a-zA-Z]{3,}\b", text.lower())
        )

        keyword_matches = query_words.intersection(text_words)
        keyword_score = len(keyword_matches) / max(len(query_words), 1)

        # Convert distance into semantic similarity
        semantic_score = 1 / (1 + distance)

        # Combined score
        final_score = (
            0.75 * semantic_score +
            0.25 * keyword_score
        )

        scored_results.append({
            "text": text,
            "page": page,
            "source": source,
            "distance": distance,
            "semantic_score": semantic_score,
            "keyword_score": keyword_score,
            "final_score": final_score,
            "keyword_matches": sorted(keyword_matches)
        })

    # Sort by combined score
    scored_results.sort(
        key=lambda x: x["final_score"],
        reverse=True
    )

    return scored_results[:top_k]


# Test hybrid retrieval
hybrid_results = hybrid_retrieve(
    "What safety precautions should be followed in a vehicle workshop?",
    top_k=5
)

for rank, result in enumerate(hybrid_results, start=1):
    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Page: {result['page']}")
    print(f"Final score: {result['final_score']:.4f}")
    print(f"Keyword matches: {result['keyword_matches']}")
    print(result["text"][:400].replace("\n", " "))
    print()

Rank: 1
Page: 11
Final score: 0.6428
Keyword matches: ['safety', 'should', 'vehicle', 'what', 'workshop']
Hazards A hazard is anything that might cause an accident or injury. Look around your workshop. You will probably see at least one hazard, perhaps more. Some can quickly be removed. Examples are: • wheels left lying on the ﬂoor, after being taken off a vehicle • oil spilt on the ﬂoor • a trolley jack handle left lying where someone could trip over it. Some hazards are always present. Warning notic

Rank: 2
Page: 10
Final score: 0.6102
Keyword matches: ['safety', 'should', 'vehicle', 'what', 'workshop']
EALTH & SAFETY Remember: all the time, be alert to prevent accidents . In any job, be aware of: • the dangers • available protection. ACTIVITY List three other causes of workshop accidents. 1 ................................................................................... 2 ................................................................................... 3 ......................

## 5.1 Hybrid Retrieval Strategy

The final retrieval stage combines semantic similarity with keyword
relevance.

Semantic retrieval identifies chunks that are conceptually related to
the user's question, while keyword matching gives additional importance
to technical terms explicitly present in the query.

The combined score is used to rank the retrieved maintenance chunks.

This approach improves retrieval precision for technical automotive
questions without requiring a more complex retrieval architecture.


In [32]:
# Retrieval evaluation queries

test_queries = [
    "What are common hazards in a vehicle workshop?",
    "What safety equipment should be worn in a workshop?",
    "How should tyres be checked for damage?",
    "What should be checked when inspecting a vehicle battery?",
    "What are common causes of brake problems?",
    "What should be checked in the suspension system?",
    "What safety precautions should be followed when using hand tools?",
    "What should be considered when replacing vehicle components?"
]

for query in test_queries:
    results = hybrid_retrieve(query, top_k=3)

    print("=" * 100)
    print(f"QUERY: {query}")

    for rank, result in enumerate(results, start=1):
        preview = result["text"].replace("\n", " ")[:220]

        print(
            f"  Rank {rank} | "
            f"Page {result['page']} | "
            f"Score {result['final_score']:.4f}"
        )
        print(f"  {preview}")

QUERY: What are common hazards in a vehicle workshop?
  Rank 1 | Page 11 | Score 0.7226
  Hazards A hazard is anything that might cause an accident or injury. Look around your workshop. You will probably see at least one hazard, perhaps more. Some can quickly be removed. Examples are: • wheels left lying on t
  Rank 2 | Page 17 | Score 0.6187
  sockets, screwdrivers, pliers, hammers, chisels and ﬁles. To work safely with them use your common sense, know which tool to use, and follow safe procedures. In a workshop the most common small injuries are cut ﬁngers or
  Rank 3 | Page 7 | Score 0.5862
  Chapter 1 Safety and good housekeeping In this chapter you will learn about: 6 Vehicle maintenance ◆ health and safety at work – what to be aware of ◆ accidents and ﬁrst-aid – what to do if something happens ◆ hazards – 
QUERY: What safety equipment should be worn in a workshop?
  Rank 1 | Page 17 | Score 0.6665
  sockets, screwdrivers, pliers, hammers, chisels and ﬁles. To work safely with the

## 6. Local Ollama LLM

The retrieved maintenance context will be passed to a local Ollama language
model.

The model is instructed to:

* Answer using the retrieved automotive maintenance context.
* Avoid inventing information that is not supported by the retrieved
  documents.
* Clearly state when the available context is insufficient.
* Provide the source page numbers used for the answer.

Using a local LLM keeps the generation stage independent of external API
services.


In [34]:
import requests

OLLAMA_URL = "http://localhost:11434"

response = requests.get(
    f"{OLLAMA_URL}/api/tags",
    timeout=10
)

response.raise_for_status()

models = response.json().get("models", [])

print("Ollama is running successfully.")
print("\nAvailable models:")

for model in models:
    print(f"- {model['name']}")

Ollama is running successfully.

Available models:
- llama3.2:3b


## Generate Grounded Answers with Ollama

The retrieved maintenance chunks will be provided to the local Ollama model as context.

The model is instructed to:

* answer using the retrieved automotive maintenance content,
* avoid inventing unsupported information,
* state when the provided context is insufficient,
* reference the relevant PDF pages used for the answer.


In [36]:
def generate_with_ollama(query, retrieved_results, model_name="llama3.2:3b"):
    context_parts = []

    for i, result in enumerate(retrieved_results, start=1):
        context_parts.append(
            f"[Source {i} | PDF Page {result['page']}]\n"
            f"{result['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are CarCare AI, an automotive maintenance assistant.

Answer the user's question using ONLY the provided maintenance context.

Rules:
1. Do not invent information that is not supported by the context.
2. If the context is insufficient, clearly say that the available document does not provide enough information.
3. Give a clear and practical answer.
4. Keep the answer focused on the user's question.
5. At the end, provide the relevant PDF page numbers as sources.

Maintenance Context:
{context}

User Question:
{query}

Answer:
"""

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": model_name,
            "prompt": prompt,
            "stream": False
        },
        timeout=120
    )

    response.raise_for_status()

    return response.json()["response"].strip()

In [37]:
query = "What are common hazards in a vehicle workshop?"

retrieved_results = hybrid_retrieve(
    query,
    top_k=5
)

answer = generate_with_ollama(
    query,
    retrieved_results
)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nSOURCES:")
for result in retrieved_results:
    print(f"- PDF Page {result['page']}")

QUESTION:
What are common hazards in a vehicle workshop?

ANSWER:
According to the maintenance context, common hazards in a vehicle workshop include:

* Wheels left lying on the floor after being taken off a vehicle
* Oil spilt on the floor
* A trolley jack handle left lying where someone could trip over it
* Brake tester rollers
* A wheel balancer
* A stand drill

These hazards can be quickly removed to minimize the risk of accidents or injuries. (Source 1 | PDF Page 11)

Additionally, rotating machinery such as wheel balancers, drills, and grindstones should have guards fitted to prevent accidents. (Source 2 | PDF Page 17)

It is also recommended to be aware of the dangers and available protection in the workshop, and to keep the workshop clean, tidy, and safe to prevent accidents. (Source 4 | PDF Page 10)

Sources:
1. PDF Page 11
2. PDF Page 17
4. PDF Page 10

SOURCES:
- PDF Page 11
- PDF Page 17
- PDF Page 7
- PDF Page 10
- PDF Page 25


## RAG Evaluation

The RAG pipeline will be tested with multiple automotive maintenance questions covering different topics from the knowledge base.

The evaluation checks whether:

* relevant maintenance information is retrieved,
* the local LLM generates grounded answers,
* answers remain focused on the retrieved document context,
* relevant PDF pages are identified as sources.


In [38]:
evaluation_queries = [
    "What are common hazards in a vehicle workshop?",
    "What safety equipment should be worn in a workshop?",
    "How should tyres be checked for damage?",
    "What should be checked when inspecting a vehicle battery?",
    "What are common causes of brake problems?",
    "What should be checked in the suspension system?",
    "What safety precautions should be followed when using hand tools?",
    "What should be considered when replacing vehicle components?",
]

for i, query in enumerate(evaluation_queries, start=1):
    print("=" * 100)
    print(f"TEST {i}")
    print(f"QUESTION: {query}")

    retrieved_results = hybrid_retrieve(
        query,
        top_k=5
    )

    answer = generate_with_ollama(
        query,
        retrieved_results
    )

    print("\nANSWER:")
    print(answer)

    print("\nRETRIEVED PAGES:")
    pages = [result["page"] for result in retrieved_results]
    print(pages)

TEST 1
QUESTION: What are common hazards in a vehicle workshop?

ANSWER:
According to the provided context, common hazards in a vehicle workshop include:

* Wheels left lying on the floor after being taken off a vehicle
* Oil spilt on the floor
* A trolley jack handle left lying where someone could trip over it
* Brake tester rollers
* A wheel balancer
* A stand drill

These hazards can be present and may require warning notices or guards to minimize the risk. It's essential to be aware of these hazards and take necessary precautions to avoid accidents.

[Source 1 | PDF Page 11]

Please note that the context only provides examples of common hazards, but does not cover all possible hazards that may exist in a vehicle workshop.

RETRIEVED PAGES:
[11, 17, 7, 10, 25]
TEST 2
QUESTION: What safety equipment should be worn in a workshop?

ANSWER:
According to the provided maintenance context, specifically from Sources 2 and 5, the recommended safety equipment to be worn in a workshop includes

## RAG Evaluation Summary

The RAG pipeline was tested across multiple automotive maintenance topics, including:

* Workshop hazards and safety
* Personal protective equipment
* Tyre inspection
* Vehicle electrical systems
* Brake systems
* Suspension systems
* Hand-tool safety
* Component replacement

The tests verify that the system can retrieve relevant maintenance content from the automotive knowledge base and generate grounded answers using the local Ollama model.


In [39]:
evaluation_summary = []

for i, query in enumerate(evaluation_queries, start=1):
    retrieved_results = hybrid_retrieve(
        query,
        top_k=5
    )

    answer = generate_with_ollama(
        query,
        retrieved_results
    )

    unique_pages = sorted(
        set(result["page"] for result in retrieved_results)
    )

    evaluation_summary.append({
        "test": i,
        "question": query,
        "retrieved_pages": unique_pages,
        "num_unique_pages": len(unique_pages),
        "answer_length": len(answer)
    })

for result in evaluation_summary:
    print("=" * 100)
    print(f"TEST {result['test']}")
    print(f"Question: {result['question']}")
    print(f"Retrieved pages: {result['retrieved_pages']}")
    print(f"Unique pages: {result['num_unique_pages']}")
    print(f"Answer length: {result['answer_length']} characters")

TEST 1
Question: What are common hazards in a vehicle workshop?
Retrieved pages: [7, 10, 11, 17, 25]
Unique pages: 5
Answer length: 425 characters
TEST 2
Question: What safety equipment should be worn in a workshop?
Retrieved pages: [9, 10, 11, 12, 17]
Unique pages: 5
Answer length: 731 characters
TEST 3
Question: How should tyres be checked for damage?
Retrieved pages: [55, 56, 59, 64, 68]
Unique pages: 5
Answer length: 584 characters
TEST 4
Question: What should be checked when inspecting a vehicle battery?
Retrieved pages: [92, 93, 94, 96]
Unique pages: 4
Answer length: 576 characters
TEST 5
Question: What are common causes of brake problems?
Retrieved pages: [97, 101, 105, 107]
Unique pages: 4
Answer length: 591 characters
TEST 6
Question: What should be checked in the suspension system?
Retrieved pages: [17, 107, 108, 110, 111]
Unique pages: 5
Answer length: 731 characters
TEST 7
Question: What safety precautions should be followed when using hand tools?
Retrieved pages: [14, 17, 

## Save RAG Configuration

The main RAG configuration and evaluation summary will be saved as a JSON file.

This records the document source, chunking settings, embedding model, vector database collection, Ollama model, and evaluation results for reproducibility.


In [40]:
rag_config = {
    "project": "CarCare AI",
    "knowledge_base": {
        "source_file": cleaned_documents[0]["source"],
        "pages": len(cleaned_documents),
        "chunks": len(chunks)
    },
    "chunking": {
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP
    },
    "embedding": {
        "model": EMBEDDING_MODEL_NAME,
        "dimension": int(embedding_model.get_sentence_embedding_dimension())
    },
    "vector_database": {
        "type": "ChromaDB",
        "collection": COLLECTION_NAME,
        "documents": collection.count()
    },
    "retrieval": {
        "method": "hybrid",
        "semantic_weight": 0.75,
        "keyword_weight": 0.25,
        "default_top_k": 5
    },
    "generation": {
        "provider": "Ollama",
        "model": "llama3.2:3b",
        "local": True
    },
    "evaluation": evaluation_summary
}

RAG_CONFIG_PATH = PROCESSED_DIR / "rag_config.json"

with open(RAG_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(rag_config, f, indent=2, ensure_ascii=False)

print("RAG configuration saved successfully.")
print(f"File: {RAG_CONFIG_PATH}")
print(f"Exists: {RAG_CONFIG_PATH.exists()}")

RAG configuration saved successfully.
File: c:\Users\EGY SKY\OneDrive\Desktop\CarCare_AI\data\processed\rag_config.json
Exists: True


C:\Users\EGY SKY\AppData\Local\Temp\ipykernel_25796\1362987698.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "dimension": int(embedding_model.get_sentence_embedding_dimension())


# Notebook 02 — RAG Pipeline Complete

The CarCare AI RAG pipeline was successfully implemented and tested.

### Completed Components

* Loaded the cleaned automotive maintenance PDF.
* Split the document into overlapping text chunks.
* Generated 384-dimensional embeddings using `all-MiniLM-L6-v2`.
* Stored the document chunks and embeddings in ChromaDB.
* Implemented semantic retrieval.
* Improved retrieval using a hybrid semantic + keyword scoring strategy.
* Connected the pipeline to a local Ollama LLM.
* Used `llama3.2:3b` for grounded answer generation.
* Preserved PDF page metadata for source references.
* Tested the system with multiple automotive maintenance questions.
* Saved the RAG configuration and evaluation results.

### RAG Architecture

```text
User Question
      ↓
Hybrid Retrieval
      ↓
ChromaDB
      ↓
Relevant PDF Chunks
      ↓
Ollama — llama3.2:3b
      ↓
Grounded Answer
      ↓
PDF Page Sources
```

### Saved Artifacts

* `data/processed/pdf_pages_cleaned.json`
* `data/processed/pdf_chunks.json`
* `data/processed/rag_config.json`
* `data/chroma_db/`

The RAG pipeline is now ready to be integrated into the CarCare AI backend.
